# GraphGenerator on ZINC

This notebook demonstrates the two-stage `GraphGenerator`: first generate a new interpretation graph, then instantiate base molecules conditionally from nearby ZINC examples.

In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from collections import Counter

from nsppk import NSPPK
from sklearn.ensemble import RandomForestClassifier

from abstractgraph.display import display, display_decomposition_graph, display_mappings
from abstractgraph.graphs import graph_to_abstract_graph
from abstractgraph.operators import *
from abstractgraph_graphicalizer.chem import ZINCLoader, draw_molecules as display_graphs
from abstractgraph_ml.estimators import GraphEstimator
from abstractgraph_generative.conditional import ConditionalAutoregressiveGenerator
from abstractgraph_generative.edge_generator import EdgeGenerator
from abstractgraph_generative.graph_generator import GraphGenerator
from abstractgraph_ml.feasibility import FeasibilityEstimator, FeasibilityEstimatorFeatureCannotExist



In [ ]:
def draw(graph, decomposition_function, *, nbits=11, label_mode="operator_hash", size=(12, 6), n_elements_per_row=8):
    ag = graph_to_abstract_graph(
        graph,
        decomposition_function=decomposition_function,
        nbits=nbits,
        label_mode=label_mode,
    )
    display(ag, size=size)
    display_mappings(ag, n_elements_per_row=n_elements_per_row)
    return ag

---

In [ ]:
loader = ZINCLoader(on_error="skip")

dataset_name = "zinc_250k"
size = 3000
min_num_nodes = 30
max_num_nodes = 50

graphs, metadata = loader.load(
    dataset_name,
    limit=size,
    min_node_count=min_num_nodes,
    max_node_count=max_num_nodes,
)

print(f"dataset: {dataset_name}")
print(f"n_graphs: {len(graphs)}")
print(f"node_range: [{min_num_nodes}, {max_num_nodes}]")


In [ ]:
label_mode = "histogram_values" #label_mode: str = "operator_hash" (default) or "histogram" or "histogram_values" for AbstractGraph node labeling.
nbits = 14
cycle_tree = add(compose(name("cycle"), cycle()), compose(name("tree"), tree()))
decomposition_function = compose(intersection_edges(), cycle_tree)

#------------------------------------------------------------------------------------------------------------------------------------------------------------------------
feasibility_kwargs = dict(
    nbits=19,
    parallel=True,
    backend="loky",
    n_jobs=-1,
)
partial_feasibility_estimators = [
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=compose(neighborhood(radius=2), unlabel()),
        **feasibility_kwargs,
    ),
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=neighborhood(radius=1),
        **feasibility_kwargs,
    ),
]
partial_feasibility_estimator = FeasibilityEstimator(partial_feasibility_estimators)

final_feasibility_estimators = [
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=compose(neighborhood(radius=2), unlabel()),
        **feasibility_kwargs,
    ),
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=neighborhood(radius=1),
        **feasibility_kwargs,
    ),
]
final_feasibility_estimator = FeasibilityEstimator(final_feasibility_estimators)
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------
edge_estimator_vectorizer = NSPPK(radius=1, distance=4, connector=0, nbits=14, dense=True, parallel=True)
edge_graph_estimator = GraphEstimator(
    transformer=edge_estimator_vectorizer,
    estimator=RandomForestClassifier(
        n_estimators=80,
        random_state=0,
        n_jobs=-1,
        class_weight="balanced_subsample",
    ),
)
edge_generator = EdgeGenerator(
    partial_feasibility_estimator=partial_feasibility_estimator,
    final_feasibility_estimator=final_feasibility_estimator,
    graph_estimator=edge_graph_estimator,
    n_negative_per_positive=3,
    n_replicates=2,
    beam_size=3,
    max_restarts=2,
    fit_n_jobs=-1,
    fit_backend="loky",
    seed=0,
)

#------------------------------------------------------------------------------------------------------------------------------------------------------------------------
context_vectorizer = NSPPK(radius=1, distance=4, connector=0, nbits=14, dense=True, parallel=True)
conditional_generator = ConditionalAutoregressiveGenerator(
    decomposition_function=decomposition_function,
    nbits=nbits,
    label_mode=label_mode,
    base_cut_radius=0,
    interpretation_cut_radius=1,
    context_vectorizer=context_vectorizer,
    n_jobs=1,
)

#------------------------------------------------------------------------------------------------------------------------------------------------------------------------
generator = GraphGenerator(
    edge_generator=edge_generator,
    conditional_generator=conditional_generator,
    seed=None,
    debug=True,
    require_new_interpretation_graph=True,
    max_same_interpretation_retries=3,
)

In [ ]:
%%time
generator.store(graphs)

In [ ]:
%%time
n_samples = 1
n_instances_per_sample = 1
generated_graphs = generator.sample(
    n_samples=n_samples,
    n_interpretation_neighbors=100,
    n_conditional_neighbors=300,
    n_instances_per_sample=n_instances_per_sample,
    interpretation_edge_removal_size=0,  # 0 bypasses edge generation; 1 removes all interpretation edges before regrowth.
    random_state=None,
    conditional_generate_kwargs=dict(
        random_state=None,
        max_backtracks=2000,
        max_attempts_per_sample=6,
        require_signature_coverage=True,
    ),
)

generated_groups = [levels[0] for levels in generator.last_level_generated_graphs_history_]
generated_interpretation_graphs = [levels[-1][0] for levels in generator.last_level_generated_graphs_history_]

print(f"generated molecules: {len(generated_graphs)}")
print("attempted seed indices:", generator.last_sampled_indices_)
print("successful seed indices:", generator.last_successful_sampled_indices_)
print("successful interpretation targets:", len(generated_interpretation_graphs))
if not generated_graphs:
    print("No molecules generated; inspect warnings and try a larger neighborhood or dataset slice.")

In [ ]:
if not generated_graphs:
    print("No generated molecules to display.")
else:
    for sample_idx, (seed_idx, seed_levels, generated_levels) in enumerate(zip(generator.last_successful_sampled_indices_, generator.last_level_seed_graphs_history_, generator.last_level_generated_graphs_history_)):
        seed_graph = seed_levels[0]
        generated_instances = generated_levels[0]
        print("seed molecule")
        display_graphs([seed_graph], n_graphs_per_line=1)
        
        print(f"generated molecule instances ({len(generated_instances)})")
        display_graphs(generated_instances, n_graphs_per_line=n_instances_per_sample)

In [ ]:
if not generated_graphs:
    print("No generated molecules to display.")
else:
    for sample_idx, (seed_idx, seed_levels, generated_levels) in enumerate(zip(generator.last_successful_sampled_indices_, generator.last_level_seed_graphs_history_, generator.last_level_generated_graphs_history_)):
        seed_graph = seed_levels[0]
        seed_interpretation_graph = seed_levels[-1]
        generated_interpretation_graph = generated_levels[-1][0]
        generated_instances = generated_levels[0]
        print("=" * 120)
        print(f"sample {sample_idx} | seed index {seed_idx}")

        print("seed molecule")
        display_graphs([seed_graph], n_graphs_per_line=1)

        print("seed interpretation graph")
        display([seed_interpretation_graph], size=(5, 4))

        print("generated interpretation graph")
        display([generated_interpretation_graph], size=(5, 4))

        print(f"generated molecule instances ({len(generated_instances)})")
        display_graphs(generated_instances, n_graphs_per_line=n_instances_per_sample)
        for generated_instance in generated_instances:
            draw(generated_instance, decomposition_function=decomposition_function, nbits=nbits, label_mode=label_mode)
